In [1]:
import pandas as pd
import xml.etree.ElementTree as ET

from IPython.display import display, HTML

PATH_ACTIONS                        = "../Data/Unprocessed/actions.xml"
PATH_ACTIONS_FILTERED               = "../Data/Processed/actions_filtered.csv"
PATH_LABEL_MAP                      = "../Data/label_map.csv"

MINIMUM_DURATION                    = 4
MINIMUM_ACTION_COUNT                = 5

In [2]:
def parse_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    records = []

    # --------------------------------------------------------------------------
    # STEP 1: Extract task-level metadata (id → source filename)
    # --------------------------------------------------------------------------
    task_sources = {}
    for task in root.findall(".//meta/project/tasks/task"):
        task_id = task.findtext("id")
        source = task.findtext("source")
        name = task.findtext("name")

        if task_id:
            task_sources[task_id] = {
                "source": source,
                "name": name
            }

    # --------------------------------------------------------------------------
    # STEP 2: Parse all tracks and link to the correct task
    # --------------------------------------------------------------------------
    for track in root.findall("track"):
        task_id = track.get("task_id")

        # Match this track to its source video via task_id
        task_info = task_sources.get(task_id, {})
        source = task_info.get("source", "unknown")
        task_name = task_info.get("name", "unknown")

        # ---- Extract track-level attributes ----
        track_attrs = {
            attr.get("name"): attr.text.strip() if attr.text else ""
            for attr in track.findall("attribute")
        }

        # ---- Extract per-frame boxes and attributes ----
        for box in track.findall("box"):
            frame = int(box.get("frame"))

            # Frame-level attributes
            frame_attrs = {
                attr.get("name"): attr.text.strip() if attr.text else ""
                for attr in box.findall("attribute")
            }

            # Merge attributes
            all_attrs = {**track_attrs, **frame_attrs}

            records.append({
                "task_name": task_name,
                "source": source,
                "frame": frame,
                **all_attrs
            })

    return pd.DataFrame(records)

In [3]:
def combine_frames(df):
    df["clip_number"] = df["clip_id"].str.extract(r"^(\d+)").astype(int)
    df["frame"] = df.groupby(["source_id", "clip_id"])["frame"].transform(lambda x: x - x.min())

    # Sort by hierarchy including numeric clip number
    df = df.sort_values(["source_id", "clip_number", "fencer", "frame"]).reset_index(drop=True)

    # Columns to group by for hierarchy
    group_cols = ["source_id", "clip_number", "clip_id", "fencer"]

    results = []

    # Iterate over groups
    for _, grp in df.groupby(group_cols):
        grp = grp.sort_values("frame").reset_index(drop=True)
        
        # Use shift/cumsum to identify consecutive runs
        grp["run"] = (grp["action"] != grp["action"].shift()).cumsum()
        
        # Aggregate start/end frames per run
        run_df = grp.groupby(["run", "action"]).agg(
            start_frame=("frame", "min"),
            end_frame=("frame", "max")
        ).reset_index(drop=False)
        
        # Add hierarchy columns
        run_df["source_id"] = grp["source_id"].iloc[0]
        run_df["clip_id"] = grp["clip_id"].iloc[0]
        run_df["fencer"] = grp["fencer"].iloc[0]
        
        # Keep only desired columns
        run_df = run_df[["source_id", "clip_id", "fencer", "action", "start_frame", "end_frame"]]
        results.append(run_df)

    # Combine all groups
    return pd.concat(results, ignore_index=True)

In [4]:
def parse_annotations(path_annotations):
    df = parse_xml(path_annotations)

    df["task_name"] = df["task_name"].str.replace("Bout ", "", regex=False)
    df["task_name"] = df["task_name"].replace("Test Upload", "1")

    df.rename(columns={"task_name": "source_id"}, inplace=True)
    df.rename(columns={"source": "clip_id"}, inplace=True)
    df.rename(columns={"ID": "fencer"}, inplace=True)
    df.rename(columns={"Action": "action"}, inplace=True)

    cols = ["source_id", "clip_id", "fencer", "action", "frame"]
    df = df[cols]

    return combine_frames(df)

In [5]:
def show_stats(df):
    def display_side_by_side(dfs: list, titles: list = None):
        html_str = ""
        for i, df in enumerate(dfs):
            title = f"<h4>{titles[i]}</h4>" if titles else ""
            html_str += f"""
            <div style="display: inline-block; vertical-align: top; margin-right: 30px;">
                {title}
                {df.to_html(index=False)}
            </div>
            """
        display(HTML(html_str))

    def get_stats(df):
        df = df.copy()
        df["duration"] = df["end_frame"] - df["start_frame"] + 1
        stats = (
            df.groupby("action")["duration"]
            .agg(["min", "max", "mean", "std"])
            .reset_index()
        )

        stats["count"] = df["action"].value_counts().reindex(stats["action"]).values
        return stats.sort_values(by="count", ascending=False)

    df_offense = df[df["action"].str.startswith("ATTACK", na=False)]
    df_defense = df[df["action"].str.startswith("DEFENSE", na=False)]

    stats_offense = get_stats(df_offense)
    stats_defense = get_stats(df_defense)

    total_offensive = len(df_offense)
    total_defensive = len(df_defense)
    total_actions = len(df)

    display_side_by_side([stats_offense, stats_defense], titles=["Offensive Action Frame Durations", "Defensive Actions Frame Durations"])

    print(f"Total Offensive Frames: {total_offensive}")
    print(f"Total Defensive Frames: {total_defensive}")

    print(f"\nTotal Frames: {total_actions}")
    print("Total classes: ", df["action"].nunique())
    print("")

In [6]:
df = parse_annotations(PATH_ACTIONS)

df["duration"] = df["end_frame"] - df["start_frame"] + 1
df_actions = df[~df["action"].isin(["ATTACK_PREPARATION", "OTHER_NO_ACTION", "OTHER_RECOVERY", 'ATTACK_FEINT'])]

show_stats(df_actions)

action,min,max,mean,std,count
ATTACK_LUNGE,2,11,7.151163,1.482983,172
ATTACK_STOP_CUT,2,15,5.067797,2.016008,59
ATTACK_STEP_CUT,3,8,5.333333,1.314257,45
ATTACK_BEAT,3,10,5.085714,1.597267,35
ATTACK_COUNTER,2,7,4.758621,1.184880,29
ATTACK_RIPOSTE,2,11,5.058824,2.135140,17
ATTACK_FLUNGE,5,11,8.200000,2.201010,10
ATTACK_REMISE,4,10,6.666667,2.338090,6
ATTACK_COMPOUND,5,17,10.000000,6.244998,3
ATTACK_SKY_HOOK,5,11,8.666667,3.214550,3


Total Offensive Frames: 379
Total Defensive Frames: 128

Total Frames: 507
Total classes:  18



In [7]:
df_actions_short = df_actions[df_actions["duration"] < MINIMUM_DURATION]
df_filtered = df_actions[~df_actions["duration"].isin(df_actions_short["duration"])].copy()

merge_map = {
    "ATTACK_STOP_CUT" : "ATTACK_BASIC",
    "ATTACK_COUNTER"  : "ATTACK_BASIC",
    "ATTACK_RIPOSTE"  : "ATTACK_BASIC",
    "ATTACK_REMISE"   : "ATTACK_BASIC",
    "ATTACK_BEAT"     : "ATTACK_BASIC",
    "ATTACK_STEP_CUT" : "ATTACK_LUNGE",
    "ATTACK_COMPUND"  : "ATTACK_LUNGE",
    "DEFENSE_PARRY_1" : "DEFENSE_PARRY",
    "DEFENSE_PARRY_2" : "DEFENSE_PARRY",
    "DEFENSE_PARRY_3" : "DEFENSE_PARRY",
    "DEFENSE_PARRY_4" : "DEFENSE_PARRY",
    "DEFENSE_PARRY_5" : "DEFENSE_PARRY",
    "DEFENSE_SKY_HOOK": "DEFENSE_DISTANCE_PULL"
}

df_filtered["action"] = df["action"].apply(lambda x: merge_map[x] if x in merge_map else x)
df_filtered = df_filtered[df_filtered.groupby("action")["action"].transform("size") >= MINIMUM_ACTION_COUNT]

show_stats(df_filtered)

df_filtered["file"] = df_filtered["source_id"] + "/" + df_filtered["clip_id"]
df_filtered.sort_values(by=["file", "fencer", "start_frame"], inplace=True)

df_filtered = df_filtered.reset_index(drop=True)
df_filtered["action_id"] = df_filtered.index.astype(int)

df_filtered = df_filtered[["file", "fencer", "action_id", "action", "start_frame", "end_frame"]]
df_filtered.to_csv(PATH_ACTIONS_FILTERED, index=False)

action,min,max,mean,std,count
ATTACK_LUNGE,4,11,6.886256,1.501222,211
ATTACK_BASIC,4,15,5.409449,1.710631,127
ATTACK_FLUNGE,5,11,8.200000,2.201010,10
action,min,max,mean,std,count
DEFENSE_DISTANCE_PULL,4,30,14.138462,6.918898,65
DEFENSE_POINT_IN_LINE,6,77,30.129032,18.242335,31
DEFENSE_PARRY,4,10,5.920000,1.411855,25


Total Offensive Frames: 348
Total Defensive Frames: 121

Total Frames: 469
Total classes:  6



In [8]:
unique_labels = sorted(df_filtered["action"].unique())

labels = pd.DataFrame(unique_labels, columns=["action"])
labels["id"] = labels.index

labels.rename(columns={"action": "label"}, inplace=True)
labels.to_csv(PATH_LABEL_MAP, index=False)

print(labels)

                   label  id
0           ATTACK_BASIC   0
1          ATTACK_FLUNGE   1
2           ATTACK_LUNGE   2
3  DEFENSE_DISTANCE_PULL   3
4          DEFENSE_PARRY   4
5  DEFENSE_POINT_IN_LINE   5
